# HRM8K dataset inspection

This standalone Colab notebook discovers every HRM8K subset and split and prints:

- subset and split names with row counts,
- columns, Hugging Face features, null counts, and Python value types,
- character and optional EXAONE token-length statistics for text columns, and
- representative raw rows without changing their structure.

The complete output is also saved as `hrm8k_dataset_report.txt` so it can be attached to Codex directly.

In [ ]:
%pip install -q -U datasets huggingface_hub transformers pandas

In [ ]:
from google.colab import userdata

HF_TOKEN = userdata.get("HF_TOKEN")
if not HF_TOKEN:
    raise RuntimeError("Add HF_TOKEN to Colab Secrets before continuing")

print("HF_TOKEN loaded from Colab Secrets")

In [ ]:
# ---- User controls ----
DATASET_ID = "HAERAE-HUB/HRM8K"
ROWS_PER_SPLIT = 3
COMPUTE_TOKEN_LENGTHS = True
TOKENIZER_ID = "LGAI-EXAONE/EXAONE-4.0-1.2B"
TOKEN_BATCH_SIZE = 256
REPORT_PATH = "/content/hrm8k_dataset_report.txt"

In [ ]:
import json
from collections import Counter
from pathlib import Path

import numpy as np
from datasets import get_dataset_config_names, get_dataset_split_names, load_dataset


def percentiles(values):
    if not values:
        return None
    array = np.asarray(values, dtype=np.float64)
    return {
        "min": int(array.min()),
        "p50": float(np.percentile(array, 50)),
        "p90": float(np.percentile(array, 90)),
        "p95": float(np.percentile(array, 95)),
        "p99": float(np.percentile(array, 99)),
        "max": int(array.max()),
        "mean": float(array.mean()),
    }


def token_lengths(tokenizer, texts):
    lengths = []
    for start in range(0, len(texts), TOKEN_BATCH_SIZE):
        batch = texts[start : start + TOKEN_BATCH_SIZE]
        encoded = tokenizer(
            batch,
            add_special_tokens=False,
            truncation=False,
            padding=False,
        )["input_ids"]
        lengths.extend(len(ids) for ids in encoded)
    return lengths


def column_summary(dataset, column, tokenizer=None):
    values = dataset[column]
    non_null = [value for value in values if value is not None]
    summary = {
        "null_count": len(values) - len(non_null),
        "python_types": dict(Counter(type(value).__name__ for value in non_null)),
    }
    if non_null and all(isinstance(value, str) for value in non_null):
        summary["nonempty_count"] = sum(bool(value.strip()) for value in non_null)
        summary["character_lengths"] = percentiles([len(value) for value in non_null])
        if tokenizer is not None:
            summary["token_lengths"] = percentiles(token_lengths(tokenizer, non_null))
        unique_count = len(set(non_null))
        summary["unique_count"] = unique_count
        if unique_count <= 20:
            summary["value_counts"] = dict(Counter(non_null).most_common())
    elif non_null and all(isinstance(value, (int, float, bool)) for value in non_null):
        numeric = np.asarray(non_null, dtype=np.float64)
        summary["numeric_min"] = float(numeric.min())
        summary["numeric_max"] = float(numeric.max())
        summary["unique_count"] = len(set(non_null))
    return summary


def likely_columns(columns):
    lowered = {column.lower(): column for column in columns}
    prompt_candidates = [
        lowered[name]
        for name in ("question", "problem", "prompt", "instruction")
        if name in lowered
    ]
    answer_candidates = [
        lowered[name]
        for name in ("answer", "solution", "response", "label", "target")
        if name in lowered
    ]
    return prompt_candidates, answer_candidates

In [ ]:
from transformers import AutoTokenizer

tokenizer = None
if COMPUTE_TOKEN_LENGTHS:
    tokenizer = AutoTokenizer.from_pretrained(TOKENIZER_ID, token=HF_TOKEN)

report = []


def emit(value=""):
    report.append(str(value))


configs = get_dataset_config_names(DATASET_ID, token=HF_TOKEN)
emit(f"Dataset: {DATASET_ID}")
emit(f"Subsets/configs ({len(configs)}): {configs}")
emit(f"Tokenizer for token statistics: {TOKENIZER_ID if tokenizer else 'disabled'}")
emit()

for config_name in configs:
    split_names = get_dataset_split_names(DATASET_ID, config_name, token=HF_TOKEN)
    emit("#" * 100)
    emit(f"SUBSET: {config_name}")
    emit(f"AVAILABLE SPLITS: {split_names}")
    emit("#" * 100)

    for split_name in split_names:
        dataset = load_dataset(
            DATASET_ID,
            config_name,
            split=split_name,
            token=HF_TOKEN,
        )
        prompt_columns, answer_columns = likely_columns(dataset.column_names)

        emit()
        emit(f"## SPLIT: {split_name}")
        emit(f"ROW COUNT: {len(dataset)}")
        emit(f"COLUMNS: {dataset.column_names}")
        emit(f"LIKELY PROMPT COLUMNS: {prompt_columns}")
        emit(f"LIKELY ANSWER COLUMNS: {answer_columns}")
        emit("FEATURES:")
        emit(json.dumps(dataset.features.to_dict(), ensure_ascii=False, indent=2, default=str))

        emit("COLUMN SUMMARIES:")
        summaries = {
            column: column_summary(dataset, column, tokenizer)
            for column in dataset.column_names
        }
        emit(json.dumps(summaries, ensure_ascii=False, indent=2, default=str))

        emit(f"REPRESENTATIVE RAW ROWS (first {min(ROWS_PER_SPLIT, len(dataset))}):")
        for index in range(min(ROWS_PER_SPLIT, len(dataset))):
            emit(f"--- ROW {index} ---")
            emit(json.dumps(dataset[index], ensure_ascii=False, indent=2, default=str))
        emit()

report_text = "\n".join(report)
Path(REPORT_PATH).write_text(report_text, encoding="utf-8")
print(report_text)
print(f"\nSaved complete report to {REPORT_PATH}")

In [ ]:
# Download the report, then attach it to Codex instead of copying a very long cell output.
from google.colab import files

files.download(REPORT_PATH)